Utilizando esse conjunto de dados: https://www.kaggle.com/datasets/disham993/9000-movies-dataset

Responda as seguintes perguntas:

In [ ]:
# Manipulação e visualização de dados
import pandas as pd
import numpy as np
import os
# Bibliotecas para aprendizado de máquina
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# MLflow para rastreamento de experimentos
import mlflow

# Supressão de avisos
import warnings
warnings.filterwarnings("ignore")

Puxar o Dataset pela API

In [ ]:
# 1️⃣ Baixar e descompactar o dataset na pasta ./data
os.makedirs("data", exist_ok=True)
!kaggle datasets download -d disham993/9000-movies-dataset -p ./data --unzip

# 2️⃣ Listar todos os arquivos baixados
print("\n📂 Arquivos na pasta data/:")
for f in os.listdir("data"):
    print(" -", f)

# 3️⃣ Procurar o primeiro arquivo CSV encontrado
csv_files = [f for f in os.listdir("data") if f.endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado na pasta 'data/'")

# 4️⃣ Ver primeiras linhas do arquivo cru (para ver o formato real)
csv_path = os.path.join("data", csv_files[0])
print(f"\n🔍 Verificando arquivo: {csv_files[0]}\n")
with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f.readline().strip())

# 5️⃣ Carregar o CSV com tolerância extra
try:
    df = pd.read_csv(
        csv_path,
        engine="python",      # mais tolerante
        on_bad_lines="skip",  # ignora linhas ruins
        encoding="utf-8",     # tenta UTF-8
        sep=","             # assume separador vírgula
    )
except Exception as e:
    print("Não conseguiu ler o arquivo")

In [ ]:
# 6️⃣ Mostrar as primeiras linhas
print(f"\n✅ Arquivo carregado: {csv_files[0]}")
df.head()

Qual tamanho do DataSet?

In [ ]:
print(f"Tamanho do DataSet: {df.size}")

Quantas linhas e colunas?

In [ ]:
print(f"Quantidade de linhas e colunas: {df.shape}")

Qual o tipo de variável de cada coluna?

In [ ]:
df.dtypes

Qual o filme com maior número de votações?

In [ ]:
# Precisei realizar a conversão da coluna Vote_count para int, para poder fazer o max(), porém ele tinha alguns dados nulos que transformei para 0 para facilitar a analise
df['Vote_Count'] = pd.to_numeric(df['Vote_Count'], errors='coerce').fillna(0).astype(int)

In [ ]:
top_movie = df.loc[df['Vote_Count'].idxmax(), ['Title', 'Vote_Count']]
print(top_movie)

Qual filme teve a maior nota (critério de desempate é o filme com mais votos)

In [ ]:
# Ajustando a coluna de nota para poder fazer uma analise melhor
df['Vote_Average'] = pd.to_numeric(df['Vote_Average'], errors='coerce').fillna(0).astype(float)

In [ ]:
# Verificando os top 10 filmes por ordem de maior nota
top10 = df.sort_values(by=['Vote_Average', 'Vote_Count'], ascending=[False, False]).head(10)
top10[['Title', 'Vote_Average', 'Vote_Count']]

In [ ]:
# Verificando de outra forma o filme com maior nota
top_average = df.loc[df['Vote_Average'].idxmax(), ['Title', 'Vote_Average']]
print(top_average)

Existem valores nulos? Se sim, qual tratamento irá realizar? (Se não temos nome de algum filme, melhor nem considerar)

In [ ]:
# Verificar valores ausentes
display(df.isnull().sum().sort_values(ascending=False))


In [ ]:
# Descartando os titulos nulos, pois não teriamos como adivinhar qual filme é
df = df.dropna(subset=['Title'])

In [ ]:
# Verificando valores ausentes após tropar titulos nulos
display(df.isnull().sum().sort_values(ascending=False))


In [ ]:
# 1️⃣ Categóricas simples
df['Genre'] = df['Genre'].fillna('Unknown')
df['Poster_Url'] = df['Poster_Url'].fillna('No poster available')

# 2️⃣ Numérica - Popularity
df['Popularity'] = df['Popularity'].fillna(df['Popularity'].median())

# 3️⃣ Categórica - Original_Language
df['Original_Language'] = df['Original_Language'].fillna(df['Original_Language'].mode()[0])


In [ ]:
# Verificando valores ausentes após tratamento das colunas
display(df.isnull().sum().sort_values(ascending=False))

Transforme as variaveis categóricas de linguagem e genero em númericas (utilize dummy)

In [ ]:
# Criar dummies para as colunas categóricas
df_dummies = pd.get_dummies(df, columns=['Original_Language', 'Genre'], drop_first=True, dtype=int)
# Verificar o resultado
df_dummies.head()


Normalize as variaveis numéricas

In [ ]:
dados_scaled_minmax = df_dummies.copy()
dados_scaled_standard = df_dummies.copy()

num_cols = df_dummies.select_dtypes(include=[np.number]).columns

minmax = MinMaxScaler()
dados_scaled_minmax[num_cols] = minmax.fit_transform(dados_scaled_minmax[num_cols])

standard = StandardScaler()
dados_scaled_standard[num_cols] = standard.fit_transform(dados_scaled_standard[num_cols])

print('Visualização após Normalização (Min-Max):')
display(dados_scaled_minmax[num_cols].describe().T.head())

print('Visualização após Padronização (Standard):')
display(dados_scaled_standard[num_cols].describe().T.head())

Armazene esses valores como um artefato dentro do MLFlow

In [ ]:
# Salvar
processed_data_path = "dados_filmes.csv"
df.to_csv(processed_data_path, index=False)
print("Dataset processado salvo localmente.")

In [ ]:
# Registrar o dataset processado como artefato no MLflow
mlflow.start_run()  # Iniciar um novo experimento
mlflow.log_artifact(processed_data_path)  # Registrar o arquivo como artefato
mlflow.end_run()  # Encerrar o experimento

print("Features armazenadas e versionadas com sucesso no MLflow!")

Quais insights é possivel obter desses dados?